# HydraNet Shared-Encoder Multi-Decoder Export

This notebook mirrors the component export flow from `notebooks/start_here.ipynb`, but swaps the single-task decoder for a task-keyed decoder collection loaded from linear-probing checkpoints.

It:
1. Loads one linear-probing student checkpoint per task.
2. Reuses the same encoder and bottleneck from one shared source task.
3. Keeps one task-specific decoder branch per task in a dictionary.
4. Exports encoder and bottleneck exactly as before.
5. Exports a single multi-decoder ONNX graph with one named output per task.

Default configuration uses `decoder_tasks`, so the exported decoder outputs are aligned to task names instead of anonymous head indices.

## 1. Parameters And Model Loading

Set `decoder_tasks` to the task decoders you want to export. Each task loads its own linear-probing checkpoint, while `encoder_source_task` selects which checkpoint provides the shared encoder and bottleneck.

In [1]:
from pathlib import Path

import hydranet

def student_signature(model):
    return {
        'n_channels': int(model.n_channels),
        'base_filters': int(model.base_filters),
        'depth': int(model.depth),
        'channel_multipliers': tuple(model.channel_multipliers),
        'channels': tuple(model.channels),
    }


# Shared-encoder / task-decoder parameters
decoder_tasks = (
    'burned_area',
    'lc',
    'roads',
)
encoder_source_task = 'burned_area'
n_shots = 5000
training = 'linear_probing'
weights_dir = '../weights'
auto_load_weights = True
checkpoint_selection = 'best'
strict = False
input_size = 224
output_dir = Path('../onnx/components_multihead')
output_dir.mkdir(parents=True, exist_ok=True)

if encoder_source_task not in decoder_tasks:
    raise ValueError('encoder_source_task must be part of decoder_tasks')

student_models = {
    task_name: hydranet.load_student(
        preset='checkpoint',
        task=task_name,
        n_shots=n_shots,
        training=training,
        auto_load_weights=auto_load_weights,
        checkpoint_selection=checkpoint_selection,
        weights_dir=weights_dir,
        strict=strict,
    )
    for task_name in decoder_tasks
}

shared_model = student_models[encoder_source_task]
shared_signature = student_signature(shared_model)
for task_name, task_model in student_models.items():
    signature = student_signature(task_model)
    assert signature == shared_signature, (task_name, signature, shared_signature)

print(f'Shared encoder task: {encoder_source_task}')
print(f'Decoder tasks: {list(decoder_tasks)}')
print(f'Shared model parameters: {sum(p.numel() for p in shared_model.parameters()):,}')

Selected best checkpoint using artifacts metrics: 20251216 (mcc_macro=0.380368194408088, f1_macro=0.5034756335849331, f1=None, acc=0.5207842295121173, best_val_loss=1.1212435603141784)
  Downloading: UNet_Myriad2_Downstream_frozen_best.pt


  Saved to: ../weights/linear_probing/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
Loading weights from: ../weights/linear_probing/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt


  Adjusted n_classes to checkpoint head: 4


Falling back to latest checkpoint because no usable artifacts metrics were found: 20260108
  Downloading: UNet_Myriad2_Downstream_frozen_best.pt


  Saved to: ../weights/linear_probing/hydranet/lc_nshot5000_frozen/lc/20260108_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
Loading weights from: ../weights/linear_probing/hydranet/lc_nshot5000_frozen/lc/20260108_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt


  Adjusted n_classes to checkpoint head: 11


Selected best checkpoint using artifacts metrics: 20251228 (mcc_macro=None, f1_macro=None, f1=0.1254076809300631, acc=0.9435345114069816, best_val_loss=4.0914110645805435)
  Downloading: UNet_Myriad2_Downstream_frozen_best.pt


  Saved to: ../weights/linear_probing/hydranet/roads_nshot5000_frozen/roads/20251228_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
Loading weights from: ../weights/linear_probing/hydranet/roads_nshot5000_frozen/roads/20251228_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
  Adjusted n_classes to checkpoint head: 1
Shared encoder task: burned_area
Decoder tasks: ['burned_area', 'lc', 'roads']
Shared model parameters: 351,172


## 2. Split Model Into Components

In [2]:
components_by_task = {
    task_name: hydranet.split_model(task_model)
    for task_name, task_model in student_models.items()
}
shared_components = components_by_task[encoder_source_task]
print(shared_components)

encoder = shared_components.encoder
bottleneck = shared_components.bottleneck
decoders_by_task = {
    task_name: task_components.decoder
    for task_name, task_components in components_by_task.items()
}

print('ENCODER:', list(encoder.keys()))
print('BOTTLENECK:', list(bottleneck.keys()))
print('DECODER TASKS:', list(decoders_by_task.keys()))
for task_name, task_decoder in decoders_by_task.items():
    print(f'{task_name}: {list(task_decoder.keys())}')

ModelComponents(
  ENCODER: 2 layer groups, 52,304 parameters (14.9%)
  BOTTLENECK: 1 layer groups, 146,816 parameters (41.8%)
  DECODER: 3 layer groups, 152,052 parameters (43.3%)
  TOTAL: 351,172 parameters
)
ENCODER: ['encoders', 'pools']
BOTTLENECK: ['bottleneck']
DECODER TASKS: ['burned_area', 'lc', 'roads']
burned_area: ['upsamplers', 'decoders', 'final_conv']
lc: ['upsamplers', 'decoders', 'final_conv']
roads: ['upsamplers', 'decoders', 'final_conv']


## 3. Export Encoder, Bottleneck, And Task-Keyed Decoder Branches To ONNX

The encoder and bottleneck exports still come from the shared source task. The decoder wrapper now uses a `ModuleDict` keyed by task name so the exported ONNX outputs stay aligned to task-specific decoder branches.

In [3]:
import copy

import torch
import torch.nn as nn

for task_model in student_models.values():
    task_model.eval()

input_channels = int(shared_model.n_channels)


class EncoderExportWrapper(nn.Module):
    def __init__(self, encoders, pools):
        super().__init__()
        self.encoders = encoders
        self.pools = pools

    def forward(self, x):
        skips = []
        current = x
        for index, encoder_block in enumerate(self.encoders):
            current = encoder_block(current)
            skips.append(current)
            if index < len(self.encoders) - 1:
                current = self.pools[index](current)
        bottleneck_in = self.pools[-1](current)
        return (bottleneck_in, *skips)


class TaskDecoderHead(nn.Module):
    def __init__(self, upsamplers, decoders, final_conv, task_name):
        super().__init__()
        self.upsamplers = copy.deepcopy(upsamplers)
        self.decoders = copy.deepcopy(decoders)
        self.final_conv = copy.deepcopy(final_conv)
        self.task_name = task_name

    def forward(self, bottleneck_out, skips):
        current = bottleneck_out
        depth = len(self.decoders)
        for index in range(depth):
            current = self.upsamplers[index](current)
            skip = skips[depth - 1 - index]
            # Keep opset 10 export free of Resize/Upsample: this model at 224x224 already matches skip sizes.
            current = torch.cat([current, skip], dim=1)
            current = self.decoders[index](current)
        return self.final_conv(current)


class MultiTaskDecoderExportWrapper(nn.Module):
    def __init__(self, decoder_components_by_task):
        super().__init__()
        if not decoder_components_by_task:
            raise ValueError('decoder_components_by_task must contain at least one task')
        self.decoder_tasks = list(decoder_components_by_task.keys())
        self.heads = nn.ModuleDict(
            {
                task_name: TaskDecoderHead(
                    decoder_components['upsamplers'],
                    decoder_components['decoders'],
                    decoder_components['final_conv'],
                    task_name=task_name,
                )
                for task_name, decoder_components in decoder_components_by_task.items()
            }
        )

    def forward(self, bottleneck_out, *skips):
        skip_list = list(skips)
        return tuple(self.heads[task_name](bottleneck_out, skip_list) for task_name in self.decoder_tasks)


encoder_wrapper = EncoderExportWrapper(encoder['encoders'], encoder['pools']).eval()
bottleneck_wrapper = bottleneck['bottleneck'].eval()
multi_task_decoder_wrapper = MultiTaskDecoderExportWrapper(decoders_by_task).eval()

dummy_input = torch.randn(1, input_channels, input_size, input_size)

with torch.no_grad():
    encoder_outputs = encoder_wrapper(dummy_input)
    bottleneck_input = encoder_outputs[0]
    skips = encoder_outputs[1:]
    bottleneck_output = bottleneck_wrapper(bottleneck_input)
    decoder_outputs = multi_task_decoder_wrapper(bottleneck_output, *skips)

assert len(decoder_outputs) == len(decoder_tasks)
decoder_outputs_by_task = dict(zip(decoder_tasks, decoder_outputs))
expected_decoder_shapes = {
    task_name: tuple(output.shape)
    for task_name, output in decoder_outputs_by_task.items()
}

component_paths = {
    'encoder': output_dir / 'encoder.onnx',
    'bottleneck': output_dir / 'bottleneck.onnx',
    'multi_task_decoder': output_dir / f'decoder_{len(decoder_tasks)}_tasks.onnx',
}

# 1) Encoder export: output = bottleneck_input + all skip tensors
torch.onnx.export(
    encoder_wrapper,
    dummy_input,
    str(component_paths['encoder']),
    export_params=True,
    opset_version=10,
    input_names=['input'],
    output_names=['bottleneck_input'] + [f'skip_{index}' for index in range(len(skips))],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'bottleneck_input': {0: 'batch_size'},
        **{f'skip_{index}': {0: 'batch_size'} for index in range(len(skips))},
    },
)

# 2) Bottleneck export: input = bottleneck_input, output = bottleneck_output
torch.onnx.export(
    bottleneck_wrapper,
    bottleneck_input,
    str(component_paths['bottleneck']),
    export_params=True,
    opset_version=10,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
)

# 3) Task-keyed decoder export: inputs = bottleneck_output + skip tensors, outputs = one tensor per task decoder
decoder_input_names = ['bottleneck_output'] + [f'skip_{index}' for index in range(len(skips))]
decoder_output_names = [f'decoder_{task_name}' for task_name in decoder_tasks]
decoder_dynamic_axes = {name: {0: 'batch_size'} for name in decoder_input_names}
decoder_dynamic_axes.update({name: {0: 'batch_size'} for name in decoder_output_names})

torch.onnx.export(
    multi_task_decoder_wrapper,
    (bottleneck_output, *skips),
    str(component_paths['multi_task_decoder']),
    export_params=True,
    opset_version=10,
    input_names=decoder_input_names,
    output_names=decoder_output_names,
    dynamic_axes=decoder_dynamic_axes,
)

print(f'Per-task output shapes: {expected_decoder_shapes}')
for name, path in component_paths.items():
    print(f'{name}: {path.resolve()}')

Per-task output shapes: {'burned_area': (1, 4, 224, 224), 'lc': (1, 11, 224, 224), 'roads': (1, 1, 224, 224)}
encoder: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/onnx/components_multihead/encoder.onnx
bottleneck: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/onnx/components_multihead/bottleneck.onnx
multi_task_decoder: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/onnx/components_multihead/decoder_3_tasks.onnx


## 4. Validate The Exported Multi-Decoder ONNX Graph

In [4]:
from collections import Counter

import onnx
from onnx import shape_inference


def tensor_shape_metadata(value_info):
    dims = []
    for dim in value_info.type.tensor_type.shape.dim:
        if dim.HasField('dim_value'):
            dims.append(dim.dim_value)
        elif dim.HasField('dim_param'):
            dims.append(dim.dim_param)
        else:
            dims.append('?')
    return dims


decoder_onnx = onnx.load(str(component_paths['multi_task_decoder']))
onnx.checker.check_model(decoder_onnx)
shape_metadata_source = 'graph output metadata'
shape_metadata_graph = decoder_onnx.graph
try:
    shape_metadata_graph = shape_inference.infer_shapes(decoder_onnx).graph
    shape_metadata_source = 'shape_inference output metadata'
except Exception as exc:
    print(f'Shape inference unavailable, falling back to graph metadata: {exc}')
decoder_opset = max(
    entry.version for entry in decoder_onnx.opset_import if entry.domain in ('', 'ai.onnx')
)
exported_input_names = [input_value.name for input_value in decoder_onnx.graph.input]
exported_output_names = [output.name for output in decoder_onnx.graph.output]
exported_output_shapes = {
    output.name: tensor_shape_metadata(output)
    for output in shape_metadata_graph.output
}
expected_output_shape_metadata = {
    f'decoder_{task_name}': ['batch_size', *expected_decoder_shapes[task_name][1:]]
    for task_name in decoder_tasks
}
op_histogram = dict(sorted(Counter(node.op_type for node in decoder_onnx.graph.node).items()))

assert decoder_opset == 10, decoder_opset
assert exported_input_names == decoder_input_names, (exported_input_names, decoder_input_names)
assert len(exported_output_names) == len(decoder_tasks), exported_output_names
assert exported_output_names == decoder_output_names, (exported_output_names, decoder_output_names)
for output_name, output_shape in exported_output_shapes.items():
    expected_shape = expected_output_shape_metadata[output_name]
    assert len(output_shape) == len(expected_shape), (
        output_name,
        output_shape,
        expected_shape,
    )
    for actual_dim, expected_dim in zip(output_shape, expected_shape):
        if actual_dim != '?':
            assert actual_dim == expected_dim, (
                output_name,
                output_shape,
                expected_shape,
            )

print('ONNX checker: PASS')
print('Structural validation: PASS (runtime validation is optional and requires onnxruntime)')
print(f'Decoder tasks: {list(decoder_tasks)}')
print(f'Decoder opset: {decoder_opset}')
print(f'Decoder inputs: {exported_input_names}')
print(f'Decoder outputs: {exported_output_names}')
print(f'Output shape metadata source: {shape_metadata_source}')
print(f'Decoder output shapes: {exported_output_shapes}')
print(f'Graph nodes: {len(decoder_onnx.graph.node)}')
print(f'Graph initializers: {len(decoder_onnx.graph.initializer)}')
print(f'Op histogram: {op_histogram}')

ONNX checker: PASS
Structural validation: PASS (runtime validation is optional and requires onnxruntime)
Decoder tasks: ['burned_area', 'lc', 'roads']
Decoder opset: 10
Decoder inputs: ['bottleneck_output', 'skip_0', 'skip_1', 'skip_2']
Decoder outputs: ['decoder_burned_area', 'decoder_lc', 'decoder_roads']
Output shape metadata source: shape_inference output metadata
Decoder output shapes: {'decoder_burned_area': ['batch_size', 4, 224, 224], 'decoder_lc': ['batch_size', 11, 224, 224], 'decoder_roads': ['batch_size', 1, 224, 224]}
Graph nodes: 166
Graph initializers: 86
Op histogram: {'Add': 18, 'Concat': 9, 'Constant': 27, 'Conv': 39, 'ConvTranspose': 9, 'Div': 9, 'Erf': 9, 'Identity': 19, 'Mul': 27}


In [5]:
import importlib.util

if importlib.util.find_spec('onnxruntime') is None:
    print('Skipping onnxruntime debug: package not installed. Install with `pip install onnxruntime`.')
else:
    try:
        bottleneck_output_np = bottleneck_output.detach().cpu().numpy()
        skip_arrays = [skip.detach().cpu().numpy() for skip in skips]
        pytorch_decoder_outputs = {
            task_name: output.detach().cpu().numpy()
            for task_name, output in decoder_outputs_by_task.items()
        }
    except NameError as exc:
        print(f'Skipping onnxruntime debug: decoder sample tensors are unavailable in this session ({exc}).')
    else:
        import numpy as np
        import onnxruntime as ort

        ort_session = ort.InferenceSession(
            str(component_paths['multi_task_decoder']),
            providers=['CPUExecutionProvider'],
        )
        ort_input_names = [input_value.name for input_value in ort_session.get_inputs()]
        ort_output_names = [output_value.name for output_value in ort_session.get_outputs()]
        ort_feeds = {'bottleneck_output': bottleneck_output_np}
        ort_feeds.update(
            {f'skip_{index}': skip_array for index, skip_array in enumerate(skip_arrays)}
        )
        ort_outputs = ort_session.run(ort_output_names, ort_feeds)
        ort_output_map = dict(zip(decoder_tasks, ort_outputs))
        ort_output_shapes = {
            task_name: tuple(output.shape)
            for task_name, output in ort_output_map.items()
        }

        assert ort_input_names == decoder_input_names, (ort_input_names, decoder_input_names)
        assert ort_output_names == decoder_output_names, (ort_output_names, decoder_output_names)
        assert len(ort_outputs) == len(decoder_tasks), len(ort_outputs)
        for task_name in decoder_tasks:
            assert ort_output_shapes[task_name] == expected_decoder_shapes[task_name], (
                task_name,
                ort_output_shapes[task_name],
                expected_decoder_shapes[task_name],
            )
            assert np.allclose(
                ort_output_map[task_name],
                pytorch_decoder_outputs[task_name],
                rtol=1e-4,
                atol=1e-5,
            ), (
                task_name,
                float(np.max(np.abs(ort_output_map[task_name] - pytorch_decoder_outputs[task_name]))),
            )

        print('onnxruntime debug: PASS')
        print(f'onnxruntime inputs: {ort_input_names}')
        print(f'onnxruntime outputs: {ort_output_names}')
        print(f'onnxruntime decoder tasks: {list(decoder_tasks)}')
        print(f'onnxruntime output shapes: {ort_output_shapes}')

onnxruntime debug: PASS
onnxruntime inputs: ['bottleneck_output', 'skip_0', 'skip_1', 'skip_2']
onnxruntime outputs: ['decoder_burned_area', 'decoder_lc', 'decoder_roads']
onnxruntime decoder tasks: ['burned_area', 'lc', 'roads']
onnxruntime output shapes: {'burned_area': (1, 4, 224, 224), 'lc': (1, 11, 224, 224), 'roads': (1, 1, 224, 224)}
